# Phase 5 — Passage à Keras sur MNIST flatten

## Objectif

Comparer le code numpy from-scratch avec l’API Keras sur un vrai dataset de référence : MNIST.

Dans cette phase, on va :
- charger MNIST ;
- aplatir les images 28x28 en vecteurs de 784 features ;
- normaliser les pixels entre 0 et 1 ;
- construire un PMC Keras ;
- entraîner le modèle ;
- évaluer les performances ;
- tracer et sauvegarder les courbes.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import time

In [2]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

X_train = X_train.reshape(-1, 784).astype("float32") / 255.0
X_test = X_test.reshape(-1, 784).astype("float32") / 255.0

print(f"Train : {X_train.shape} | Test : {X_test.shape}")
print(f"Classes uniques : {np.unique(y_train)}")

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
Train : (60000, 784) | Test : (10000, 784)
Classes uniques : [0 1 2 3 4 5 6 7 8 9]


## 1. Modèle Keras

Architecture demandée :
- Dense(128, relu)
- Dense(64, relu)
- Dense(10, softmax)

avec `input_shape=(784,)` sur la première couche.

In [3]:
tf.random.set_seed(42)

model = keras.Sequential([
    keras.layers.Dense(128, activation="relu", input_shape=(784,)),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(10, activation="softmax")
])

model.summary()

d:\Cours\5ème Année IPSSI - Paris\Machine Learning\deeplearning-j1-pmc-from-scratch-DONOU-Serge\.venv311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │       100,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │           650 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 109,386 (427.29 KB)

 Trainable params: 109,386 (427.29 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

## 2. Entraînement

Paramètres demandés :
- `epochs = 5`
- `batch_size = 64`
- `validation_split = 0.1`

In [5]:
start = time.time()

history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.1,
    verbose=1
)

train_time = time.time() - start
print(f"\nTemps d'entraînement : {train_time:.2f} secondes")

Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.9150 - loss: 0.2933 - val_accuracy: 0.9607 - val_loss: 0.1270
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9648 - loss: 0.1188 - val_accuracy: 0.9692 - val_loss: 0.0936
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9759 - loss: 0.0818 - val_accuracy: 0.9723 - val_loss: 0.0893
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.9829 - loss: 0.0589 - val_accuracy: 0.9728 - val_loss: 0.0873
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - accuracy: 0.9870 - loss: 0.0439 - val_accuracy: 0.9742 - val_loss: 0.0888

Temps d'entraînement : 26.59 secondes


In [6]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)

print(f"Test loss     : {test_loss:.4f}")
print(f"Test accuracy : {test_acc:.2%}")

Test loss     : 0.0872
Test accuracy : 97.43%


In [7]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history["loss"], label="train_loss")
axes[0].plot(history.history["val_loss"], label="val_loss")
axes[0].set_title("Courbe de loss - MNIST Keras")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(history.history["accuracy"], label="train_acc")
axes[1].plot(history.history["val_accuracy"], label="val_acc")
axes[1].set_title("Courbe d'accuracy - MNIST Keras")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.savefig("phase5_mnist_curves.png", dpi=100, bbox_inches="tight")
plt.show()

print("Figure sauvegardée : phase5_mnist_curves.png")

Figure sauvegardée : phase5_mnist_curves.png


C:\Users\serge\AppData\Local\Temp\ipykernel_31448\1737677845.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Vérification attendue

### Scénario normal
On doit observer :
- une accuracy test autour de 97% à 98% ;
- une loss de validation qui descend correctement ;
- des courbes propres et stables ;
- une image sauvegardée : `phase5_mnist_curves.png`.

### Interprétation
Par rapport au code numpy from-scratch :
- beaucoup moins de lignes de code ;
- moins de risques de bug dans la backprop ;
- de meilleures performances sur un vrai dataset.

### Conclusion
Keras permet de retrouver la même logique qu’en from-scratch, mais avec une API beaucoup plus compacte et fiable.